# DataFog PII-NER v1 — Smoke Test

Validates that the full model architecture is wired correctly by overfitting on 100 examples.

**Success criteria:**
- Training loss < 0.1
- F1 on training data > 0.95
- Runs in under 5 minutes on a T4 GPU

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU** (or any GPU)
2. Run all cells

## 1. Setup

In [ ]:
# Clone the repo and install the package
!git clone https://github.com/DataFog/datafog-labs.git /content/datafog-labs 2>/dev/null || (cd /content/datafog-labs && git pull)
!pip install -e "/content/datafog-labs/pii-ner-v1[dev]" -q

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    try:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
        print(f"Memory: {mem / 1e9:.1f} GB")
    except Exception:
        print("Memory: (could not detect)")
else:
    print("No GPU found — this will run on CPU (slower but still works)")

## 2. Load Model

In [ ]:
from datafog_pii_ner.model import PiiNerConfig, PiiNerModel
from datafog_pii_ner.data.label_schema import NUM_LABELS, ID_TO_LABEL

config = PiiNerConfig(
    backbone="microsoft/deberta-v3-xsmall",
    num_labels=NUM_LABELS,
)
model = PiiNerModel(config)

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {param_count:,}")
print(f"Trainable parameters: {trainable_count:,}")
print(f"BIO label count:      {NUM_LABELS}")

## 3. Load Data (100 examples)

In [ ]:
from transformers import AutoTokenizer
from datafog_pii_ner.data.dataset import load_single_dataset

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-xsmall")

dataset = load_single_dataset(
    dataset_name="ai4privacy",
    tokenizer=tokenizer,
    max_seq_len=256,
    max_char_len=20,
    max_examples=100,
)

print(f"Loaded {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print(f"Sample input_ids length: {len(dataset[0]['input_ids'])}")
print(f"Sample char_ids shape: {len(dataset[0]['char_ids'])} x {len(dataset[0]['char_ids'][0])}")

## 4. Train (overfit)

In [ ]:
from transformers import TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.training.metrics import compute_metrics
from datafog_pii_ner.training.train import PiiTrainer

collator = PiiDataCollator(tokenizer=tokenizer, max_char_len=20)

training_args = TrainingArguments(
    output_dir="/content/smoke_test_output",
    num_train_epochs=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-4,
    warmup_steps=0,
    weight_decay=0.0,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

trainer = PiiTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,  # Same data — intentional overfit
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Starting training...")
train_result = trainer.train()
print(f"\nFinal training loss: {train_result.training_loss:.4f}")

## 5. Evaluate

In [ ]:
eval_results = trainer.evaluate()

print("=" * 50)
print("SMOKE TEST RESULTS")
print("=" * 50)
print(f"  Training loss:   {train_result.training_loss:.4f}")
print(f"  F1:              {eval_results.get('eval_overall_f1', 0):.4f}")
print(f"  Precision:       {eval_results.get('eval_overall_precision', 0):.4f}")
print(f"  Recall:          {eval_results.get('eval_overall_recall', 0):.4f}")

# Tier recalls
for tier in [1, 2, 3, 4]:
    key = f"eval_tier_{tier}_recall"
    if key in eval_results:
        print(f"  Tier {tier} recall:  {eval_results[key]:.4f}")

print()
loss_ok = train_result.training_loss < 0.1
f1_ok = eval_results.get('eval_overall_f1', 0) > 0.95
print(f"  Loss < 0.1:  {'PASS ✓' if loss_ok else 'FAIL ✗'} ({train_result.training_loss:.4f})")
print(f"  F1 > 0.95:   {'PASS ✓' if f1_ok else 'FAIL ✗'} ({eval_results.get('eval_overall_f1', 0):.4f})")
print()
if loss_ok and f1_ok:
    print("  ✓ SMOKE TEST PASSED — model is wired correctly!")
else:
    print("  ✗ SMOKE TEST FAILED — check model wiring")

## 6. Sample Predictions

In [ ]:
predictions = trainer.predict(dataset.select(range(5)))

for i in range(5):
    pred_ids = predictions.predictions[i]
    label_ids = predictions.label_ids[i]
    input_ids = dataset[i]["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    print(f"\n{'='*60}")
    print(f"Example {i + 1}")
    print(f"{'='*60}")
    print(f"{'Token':<25} {'Predicted':<25} {'Ground Truth':<25} {'Match'}")
    print(f"{'-'*25} {'-'*25} {'-'*25} {'-'*5}")

    for tok, pred, label in zip(tokens, pred_ids, label_ids):
        if label == -100:
            continue
        pred_label = ID_TO_LABEL.get(int(pred), "O")
        true_label = ID_TO_LABEL.get(int(label), "O")
        if true_label != "O" or pred_label != "O":
            match = "✓" if pred_label == true_label else "✗"
            print(f"{tok:<25} {pred_label:<25} {true_label:<25} {match}")

## Next Steps

If the smoke test passes:
1. **Full training** — Load all 360K examples, train for 10 epochs with differential learning rates
2. **Benchmark** — Evaluate against GLiNER2 and Presidio baselines
3. **ONNX export** — Quantize to INT8 for production deployment